# Expérience 03 Tokenizer par défaut + Full Fine-Tuning

**Modèle :** `SmolLM3-3B-Base` · **Données :** wolof monolingue · **Méthode :** full bfloat16

## Rôle de cette expérience

Elle sert deux comparaisons :

| Face à | Ce qu'elle mesure |
|---|---|
| Expérience 04 (étendu + full) | l'effet du **tokenizer**, à méthode constante |
| Expérience 01 (défaut + LoRA) | l'effet de la **méthode**, à tokenizer constant |


## 1  Environnement

In [ ]:
!pip install -q -U "transformers>=4.53.0" accelerate datasets pandas matplotlib peft bitsandbytes

## 2 Configuration et chargement

In [ ]:
ROOT       = "/content/drive/MyDrive/SmolLM3-3B-Base"
RUN        = "wolof-default-full"
GPU        = "H100-80Go"
METHODE    = "full"

MODEL_PATH = "HuggingFaceTB/SmolLM3-3B-Base"
TOK_PATH   = "HuggingFaceTB/SmolLM3-3B-Base"
OUT        = f"{ROOT}/runs/{RUN}"

MAX_STEPS  = 2234       
SEQ_LEN    = 2048
MICRO_BS   = 4
GRAD_ACC   = 8          
LR         = 2e-5        
SEED       = 42
MEM_FRAC   = 0.8
OPTIM      = "adamw_torch"

TOKENS_PAR_PAS = MICRO_BS * GRAD_ACC * SEQ_LEN
print(f"{TOKENS_PAR_PAS:,} tokens/pas | {MAX_STEPS} pas = {MAX_STEPS*TOKENS_PAR_PAS/1e6:.0f} M tokens")

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

if torch.cuda.is_available():
    torch.cuda.set_per_process_memory_fraction(MEM_FRAC, 0)
    tot = torch.cuda.get_device_properties(0).total_memory / 1e9
    budget = tot * MEM_FRAC
    print(f"{torch.cuda.get_device_name(0)} | budget {budget:.0f} Go")
    if budget < 45:
        print("ATTENTION : budget serré — envisager OPTIM = 'adamw_bnb_8bit'")

try:
    import flash_attn
    ATTN = "flash_attention_2"
except ImportError:
    ATTN = "sdpa"
    print("flash_attn absent → sdpa")

tok = AutoTokenizer.from_pretrained(TOK_PATH)
if tok.pad_token is None: tok.pad_token = tok.eos_token

model = AutoModelForCausalLM.from_pretrained(MODEL_PATH, dtype=torch.bfloat16,
                                             device_map={"": 0}, attn_implementation=ATTN)
model.config.use_cache = False
model.gradient_checkpointing_enable()

n_e = sum(p.numel() for p in model.parameters() if p.requires_grad)
n_t = sum(p.numel() for p in model.parameters())
print(f"paramètres entraînables : {n_e:,} / {n_t:,} ({100*n_e/n_t:.0f} %)")

## 3  Données

In [ ]:
import pandas as pd
from datasets import Dataset

train_df = pd.read_parquet(f"{ROOT}/data/wolof_bpe_train_v1.parquet")
eval_df  = pd.read_parquet(f"{ROOT}/data/wolof_eval_clean_v1.parquet")

TEXTES_TRAIN = train_df["text"].astype(str).tolist()
TEXTES_EVAL  = eval_df["text"].astype(str).tolist()
print(f"train : {len(TEXTES_TRAIN):,} | éval propre : {len(TEXTES_EVAL):,}")

def paqueter(textes, tokenizer, longueur):
    ids = []
    for t in textes:
        ids.extend(tokenizer.encode(t, add_special_tokens=False) + [tokenizer.eos_token_id])
    n = (len(ids) // longueur) * longueur
    blocs = [ids[i:i+longueur] for i in range(0, n, longueur)]
    return Dataset.from_dict({"input_ids": blocs, "labels": [b[:] for b in blocs]})

ds_train = paqueter(TEXTES_TRAIN, tok, SEQ_LEN).shuffle(seed=SEED)
ds_eval  = paqueter(TEXTES_EVAL,  tok, SEQ_LEN)
print(f"blocs — train {len(ds_train):,} | éval {len(ds_eval):,}")
print(f"epochs équivalent : {MAX_STEPS*MICRO_BS*GRAD_ACC/len(ds_train):.2f}")

## 4  BPB avant entraînement

In [ ]:
import math, torch
from torch.nn import functional as F

@torch.no_grad()
def compute_bpb(model, tokenizer, textes, max_len=1024):
    etait = model.training
    model.eval()
    nats = octets = tokens = 0
    for t in textes:
        t = str(t)
        b = len(t.encode("utf-8"))
        if b == 0: continue
        ids = tokenizer.encode(t, add_special_tokens=False)[:max_len]
        if len(ids) < 2: continue
        x = torch.tensor([ids], device="cuda:0")
        logits = model(x).logits[0, :-1]
        nats   += F.cross_entropy(logits.float(), x[0, 1:], reduction="sum").item()
        octets += b
        tokens += len(ids) - 1
    if etait: model.train()
    return {"bpb": nats/(math.log(2)*octets), "ppl": math.exp(nats/tokens),
            "n_octets": octets, "n_tokens": tokens}

bpb_avant = compute_bpb(model, tok, TEXTES_EVAL[:500])
print(f"BPB avant : {bpb_avant['bpb']:.4f} | PPL : {bpb_avant['ppl']:.2f}")

## 5  Entraînement


In [ ]:
import time
from transformers import Trainer, TrainingArguments, default_data_collator

HP = {"lr": LR, "optim": OPTIM, "scheduler": "cosine", "warmup_ratio": 0.03,
      "seq_len": SEQ_LEN, "micro_bs": MICRO_BS, "grad_acc": GRAD_ACC,
      "batch_effectif": MICRO_BS*GRAD_ACC, "max_steps": MAX_STEPS, "seed": SEED,
      "attention": ATTN, "gradient_checkpointing": True}

args = TrainingArguments(
    output_dir=OUT, max_steps=MAX_STEPS,
    per_device_train_batch_size=MICRO_BS, gradient_accumulation_steps=GRAD_ACC,
    learning_rate=LR, lr_scheduler_type="cosine", warmup_ratio=0.03,
    logging_steps=25, eval_strategy="steps", eval_steps=100,
    save_steps=500, save_total_limit=3,
    bf16=True, fp16=False, gradient_checkpointing=True,
    optim=OPTIM, report_to="none", seed=SEED,
)
trainer = Trainer(model=model, args=args, train_dataset=ds_train,
                  eval_dataset=ds_eval, data_collator=default_data_collator)

t0 = time.time()
res = trainer.train()
duree = (time.time() - t0) / 3600
print(res.metrics)
print(f"\nterminé en {duree:.2f} h")

## 6 Évaluation finale

In [ ]:
bpb_apres = compute_bpb(model, tok, TEXTES_EVAL[:500])
print(f"BPB : {bpb_avant['bpb']:.4f} → {bpb_apres['bpb']:.4f}  "
      f"(gain {100*(1-bpb_apres['bpb']/bpb_avant['bpb']):.1f} %)")
print(f"PPL : {bpb_avant['ppl']:.2f} → {bpb_apres['ppl']:.2f}")

In [ ]:
import matplotlib.pyplot as plt

hist = trainer.state.log_history
ltr  = [(h["step"], h["loss"])      for h in hist if "loss" in h]
lval = [(h["step"], h["eval_loss"]) for h in hist if "eval_loss" in h]
llr  = [(h["step"], h["learning_rate"]) for h in hist if "learning_rate" in h]

fig, ax = plt.subplots(1, 2, figsize=(14, 4.5))
s, l = zip(*ltr)
ax[0].plot(s, l, marker="o", ms=3, color="#1f77b4", label="train")
if lval:
    sv, lv = zip(*lval)
    ax[0].plot(sv, lv, marker="s", ms=4, color="#d62728", lw=2, label="validation")
ax[0].set_xlabel("pas"); ax[0].set_ylabel("loss")
ax[0].set_title(f"{RUN} — loss"); ax[0].legend(); ax[0].grid(alpha=.3)

if llr:
    sl, vl = zip(*llr)
    ax[1].plot(sl, vl, color="darkorange", lw=2)
ax[1].set_xlabel("pas"); ax[1].set_ylabel("learning rate")
ax[1].set_title("scheduler cosine"); ax[1].grid(alpha=.3)

plt.tight_layout()
plt.savefig(f"{OUT}/courbes.png", dpi=200, bbox_inches="tight")
plt.show()

print(f"loss train : {ltr[0][1]:.3f} → {ltr[-1][1]:.3f}")
if lval: print(f"loss val   : {lval[-1][1]:.3f}")

## 7  Journal du run

In [ ]:
import json, os
os.makedirs(OUT, exist_ok=True)

journal = {
    "run": RUN, "gpu": GPU, "modele": MODEL_PATH, "tokenizer": TOK_PATH,
    "vocab": len(tok), "donnees": "wolof monolingue",
    "methode": METHODE, "cadrage": "donnees egales (1 epoch)",
    "hyperparametres": HP,
    "budget": {"tokens_vus_M": MAX_STEPS*TOKENS_PAR_PAS/1e6,
               "duree_heures": round(duree, 3),
               "epochs_equivalent": round(MAX_STEPS*MICRO_BS*GRAD_ACC/len(ds_train), 3)},
    "loss": {"train_premiere": ltr[0][1], "train_derniere": ltr[-1][1],
             "val_derniere": lval[-1][1] if lval else None,
             "moyenne_run": res.metrics.get("train_loss"),
             "courbe_train": ltr, "courbe_val": lval},
    "eval": {"split": "wolof_eval_clean_v1.parquet",
             "bpb_avant": bpb_avant["bpb"], "bpb_apres": bpb_apres["bpb"],
             "ppl_avant": bpb_avant["ppl"], "ppl_apres": bpb_apres["ppl"]},
}
json.dump(journal, open(f"{OUT}/journal.json", "w"), indent=2, ensure_ascii=False)

model.save_pretrained(f"{OUT}/model", safe_serialization=True)
tok.save_pretrained(f"{OUT}/model")
print("sauvegardé :", OUT)